In [139]:
import numpy as np

In [140]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [141]:
class Particle: 
    swarm_best_position = None
    swarm_best_fitness = float('-inf')
    
    def __init__(self, X_train, y_train, calc_accuracy_fn, alpha, c_inertia, c_social, c_cognitive):
        self.X_train = X_train
        self.y_train = y_train
        self.calc_accuracy_fn = calc_accuracy_fn
        self.alpha = alpha
        
        self.c_inertia = c_inertia
        self.c_social = c_social
        self.c_cognitive = c_cognitive
        self.position = self.initialize_position()
        self.velocity = self.initialize_velocity(self.X_train.shape[1])

        self.fitness = self.calc_fitness()
        self.personal_best_position = self.position.copy()
        self.personal_best_fitness = self.fitness

        if Particle.swarm_best_position is None or Particle.swarm_best_fitness < self.fitness:
            Particle.swarm_best_fitness = self.fitness
            Particle.swarm_best_position = self.position.copy()
        
    def calc_fitness(self): 
        if not any(self.position):
            return float('-inf')
        acc = self.calc_accuracy_fn(self.position, self.X_train, self.y_train)
        num_features = sum(self.position)
        return self.alpha * acc + (1 - self.alpha) * (1 - num_features / self.X_train.shape[1])
    
    def initialize_position(self):
        return np.random.choice([True,False], size = self.X_train.shape[1])
    
    def initialize_velocity(self, num_dimensions):
        return np.random.uniform(-1, 1, size=num_dimensions)
    
    def update_velocity(self):
        r_s = np.random.random(len(self.position))
        r_c = np.random.random(len(self.position))
        assert self.swarm_best_position is not None
        
        social_velocity = self.c_social * r_s * ((self.swarm_best_position.astype(float)) - self.position.astype(float))
        cognitive_velocity = self.c_cognitive * r_c * ((self.swarm_best_position.astype(float)) - self.position.astype(float))
        
        self.velocity = self.c_inertia * self.velocity + social_velocity + cognitive_velocity
        self.velocity = np.clip(self.velocity, -4, 4)
        
    def move(self):
        self.update_velocity();
        probabilities = sigmoid(self.velocity)
        random_vector = np.random.rand(len(probabilities))
        self.position = random_vector < probabilities
        
        self.fitness = self.calc_fitness()
        if self.fitness > self.personal_best_fitness:
            self.personal_best_position = self.position.copy()
            self.personal_best_fitness = self.fitness
            if self.fitness > Particle.swarm_best_fitness:
                Particle.swarm_best_position = self.position.copy()
                Particle.swarm_best_fitness = self.fitness

In [142]:
def BPSO(X_train, y_train, calc_accuracy_fn, alpha, num_iters, swarm_size, c_inertia, c_social, c_cognitive):
    swarm = [Particle(X_train, y_train, calc_accuracy_fn, alpha, c_inertia, c_social, c_cognitive) for _ in range(swarm_size)]
    for _ in range(num_iters):
        for p in swarm:
            p.move()
    return Particle.swarm_best_position, Particle.swarm_best_fitness